# Etapa de Ingeniería de atributos

In [ ]:
"""
SCRIPT DE INGENIERÍA DE ATRIBUTOS PARA DATASET DE DENGUE
--------------------------------------------------------
Este script aplica transformaciones de ingeniería de atributos al dataset
original sin crear rezagos temporales ni reducción dimensional.
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, RobustScaler

# ============================================================
# CONFIGURACIÓN
# ============================================================
RUTA_ORIGEN = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\1_red_neuronal_1_sklearn_mlp\2_datos\1_raw\2_meteo_epi_2021-2026_1_rezagos.xlsx"

RUTA_DESTINO = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\1_red_neuronal_1_sklearn_mlp\1_scripts\modelo_red_neuronal_MLP_y_secuencia_aprendizaje\1_dataset_con_ingenieria_atributos"

NOMBRE_ARCHIVO_SALIDA = "dataset_dengue_con_ingenieria_atributos.xlsx"

VARIABLES_BASE = [
    "temp", "temp_max", "temp_min", "hum_esp", "hum_rel",
    "prec", "dias_lluvia", "vel_vi", "vel_vi_max", "vel_vi_min",
    "soi", "sst",
]

# ============================================================
# 1. CARGA Y LIMPIEZA INICIAL
# ============================================================
print("=== CARGA DE DATOS ===")
df = pd.read_excel(RUTA_ORIGEN)
df["fecha"] = pd.to_datetime(df["fecha"])
df = df.sort_values("fecha").reset_index(drop=True)

print(f"Filas cargadas: {len(df)}")
print(f"Columnas originales: {df.shape[1]}")
print(f"Rango de fechas: {df['fecha'].min().date()} a {df['fecha'].max().date()}")

# ============================================================
# 2. LIMPIEZA DE OUTLIERS EN SOI (como se indica en el enunciado)
# ============================================================
print("\n=== LIMPIEZA DE SOI ===")
UMBRAL_SOI = 50
soi_cols = ["soi"] + [f"soi_lag_{i}" for i in range(1, 13)]  # Hasta lag 12

n_afectados = (df[soi_cols].abs() > UMBRAL_SOI).sum().sum()
print(f"Celdas de SOI fuera de rango detectadas: {n_afectados}")

if n_afectados > 0:
    for c in soi_cols:
        df.loc[df[c].abs() > UMBRAL_SOI, c] = np.nan
    df[soi_cols] = df[soi_cols].interpolate(method="linear", limit_direction="both")
    print("Valores de SOI fuera de rango reemplazados por interpolación lineal.")
else:
    print("No se detectaron valores de SOI fuera de rango.")

# ============================================================
# 3. INGENIERÍA DE ATRIBUTOS
# ============================================================
print("\n=== INGENIERÍA DE ATRIBUTOS ===")

# 3.1. Variables derivadas del clima (interacciones)
print("  - Creando variables derivadas del clima...")

# Amplitud térmica diaria
df["amplitud_termica"] = df["temp_max"] - df["temp_min"]

# Humedad absoluta estimada (presión de vapor) - aproximación simple
# Usando la fórmula de Magnus: e = 6.112 * exp((17.67*T)/(T+243.5))
# con T en °C, resultado en hPa
df["presion_vapor"] = 6.112 * np.exp((17.67 * df["temp"]) / (df["temp"] + 243.5))

# Punto de rocío aproximado (usando fórmula inversa de Magnus)
df["punto_rocío"] = (243.5 * np.log(df["presion_vapor"]/6.112)) / (17.67 - np.log(df["presion_vapor"]/6.112))

# 3.2. Variables estacionales (capturan ciclos anuales)
print("  - Creando variables estacionales...")
df["dia_del_año"] = df["fecha"].dt.dayofyear

# Seno y coseno del día del año para capturar estacionalidad anual
df["sen_estacional"] = np.sin(2 * np.pi * df["dia_del_año"] / 365.25)
df["cos_estacional"] = np.cos(2 * np.pi * df["dia_del_año"] / 365.25)

# Número de semana epidemiológica
df["semana_epi"] = df["fecha"].dt.isocalendar().week

# 3.3. Variables de tendencia temporal
print("  - Creando variables de tendencia temporal...")
df["tiempo_index"] = np.arange(len(df))  # Índice secuencial
df["tiempo_index_cuadrado"] = df["tiempo_index"] ** 2  # Captura tendencias no lineales

# 3.4. Variables climáticas acumuladas (ventanas móviles simples)
print("  - Creando variables climáticas acumuladas...")
# No usamos rolling para no crear dependencia temporal en el script actual,
# pero podemos crear agregados simples por semana epidemiológica o mes

# 3.5. Variables de interacción clima-tiempo
print("  - Creando interacciones clima-tiempo...")
df["temp_tiempo_interaccion"] = df["temp"] * df["tiempo_index"] / 100
df["prec_tiempo_interaccion"] = df["prec"] * df["tiempo_index"] / 100

# 3.6. Variables de proporción y ratios
print("  - Creando variables de ratios...")
# Evitamos división por cero
df["prec_por_dia_lluvia"] = df["prec"] / (df["dias_lluvia"] + 1e-6)
df["hum_esp_por_temp"] = df["hum_esp"] / (df["temp"] + 273.15)  # Temperatura en Kelvin

# 3.7. Variables categóricas codificadas
print("  - Codificando variables categóricas...")
df["temporada"] = df["fecha"].dt.month.map({
    1: "Verano", 2: "Verano", 3: "Otoño",
    4: "Otoño", 5: "Otoño", 6: "Invierno",
    7: "Invierno", 8: "Invierno", 9: "Primavera",
    10: "Primavera", 11: "Primavera", 12: "Verano"
})

# Codificación one-hot de temporada
df = pd.get_dummies(df, columns=["temporada"], prefix="temporada")

# 3.8. Variables polinómicas básicas (términos cuadráticos)
print("  - Creando términos polinómicos...")
for var in ["temp", "hum_rel", "prec"]:
    df[f"{var}_cuadrado"] = df[var] ** 2

# ============================================================
# 4. VERIFICACIÓN DE NULOS Y CALIDAD
# ============================================================
print("\n=== VERIFICACIÓN DE CALIDAD ===")
nulos_totales = df.isna().sum().sum()
print(f"Total de valores nulos después de la ingeniería: {nulos_totales}")

if nulos_totales > 0:
    print("Valores nulos por columna:")
    nulos_por_columna = df.isna().sum()
    print(nulos_por_columna[nulos_por_columna > 0])

    # Imputación simple para valores nulos restantes
    print("Imputando valores nulos restantes con la mediana...")
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64'] and df[col].isna().any():
            df[col].fillna(df[col].median(), inplace=True)

print(f"Dimensiones finales del dataset: {df.shape[0]} filas, {df.shape[1]} columnas")

# ============================================================
# 5. GUARDADO DEL DATASET TRANSFORMADO
# ============================================================
print("\n=== GUARDADO DEL DATASET ===")
import os

# Crear directorio si no existe
os.makedirs(RUTA_DESTINO, exist_ok=True)

ruta_completa = os.path.join(RUTA_DESTINO, NOMBRE_ARCHIVO_SALIDA)
df.to_excel(ruta_completa, index=False)
print(f"Dataset guardado en: {ruta_completa}")

# ============================================================
# 6. RESUMEN DE VARIABLES CREADAS
# ============================================================
print("\n=== RESUMEN DE VARIABLES CREADAS ===")
variables_originales = set(VARIABLES_BASE + ["casos_dengue", "fecha"])
variables_nuevas = set(df.columns) - variables_originales

print(f"Variables originales (sin contar rezagos): {len(variables_originales)}")
print(f"Variables nuevas creadas: {len(variables_nuevas)}")
print(f"Total de variables en el dataset: {df.shape[1]}")

print("\nVariables nuevas:")
for i, var in enumerate(sorted(variables_nuevas), 1):
    print(f"  {i:2d}. {var}")

print("\n" + "="*60)
print("✅ ¡Ingeniería de atributos completada exitosamente!")
print("="*60)